# 05 · Cross-scale coupling (Appendix B.1)

Sensitivity of each velocity head of WFM 3-scale (haar, TRL) to a perturbation of the
interpolant at one scale. Independent per-scale flows would give a diagonal matrix.
No rollout and no retraining: forward passes only, about a minute.

Writes `artifacts/tables/raw/cross_scale_sensitivity_trl_wfm3_haar.{json,npz}` and
`artifacts/figures/diag_cross_scale_sensitivity_trl.pdf` (not in the paper). The LaTeX table is built from the JSON
by `01_tables.ipynb` (`P.s2_cross_scale()`, Table S2).

In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import shutil
import fsspec
import torch
import yaml
import sys

from pathlib import Path as _Path
PROJECT_ROOT = str(next(p for p in [_Path.cwd(), *_Path.cwd().parents] if (p / "pyproject.toml").exists()))  # repo root, works on laptop and cluster
sys.path.insert(0, PROJECT_ROOT)

from wavelet_flow_matching.utils.well_data import get_well_dataset, get_well_inputs
from wavelet_flow_matching.modules.wavelet_fm import WaveletFlowModule
from wavelet_flow_matching.utils import cross_scale_utils as csu

rc = {
    "axes.axisbelow": True,
    "axes.linewidth": 1.0,
    "axes.labelsize": "x-small",
    "axes.titlesize": "x-small",
    "figure.dpi": 150,
    "figure.figsize": (3.2, 3.2),
    "font.size": 12.0,
    "legend.fontsize": "xx-small",
    "lines.linewidth": 1.0,
    "savefig.transparent": True,
    "xtick.labelsize": "xx-small",
    "xtick.major.width": 1.0,
    "ytick.labelsize": "xx-small",
    "ytick.major.width": 1.0,
    "font.family": ["serif"],
    "text.usetex": shutil.which("latex") is not None,   # same LaTeX font as figures.ipynb
    "text.latex.preamble": r"\usepackage{amsfonts}",
}
plt.rcParams.update(rc)

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print("Imports OK | device:", device)

## 1 · Checkpoint and data

In [ ]:
well_base_path    = f"{PROJECT_ROOT}/data/datasets"
well_dataset_name = 'turbulent_radiative_layer_2D'
stats_path        = f"{well_base_path}/{well_dataset_name}/stats.yaml"

ckpt_3scale = f'{PROJECT_ROOT}/experiments/{well_dataset_name}/ckpts/WaveletFlow_{well_dataset_name}_3scales_haar_v4.0_20260429_200656-epoch=181-train_loss=0.0191-val_loss=0.0228_portable.ckpt'



OUT_RAW = _Path(PROJECT_ROOT) / "artifacts" / "tables" / "raw"
OUT_FIG = _Path(PROJECT_ROOT) / "artifacts" / "figures"
OUT_RAW.mkdir(parents=True, exist_ok=True)

In [ ]:
with fsspec.open(stats_path, mode='r') as f:
    stats = yaml.safe_load(f)

model = WaveletFlowModule.from_checkpoint(ckpt_3scale, map_location=device)
#model = WaveletFlowModule.from_checkpoint(ckpt_1scale, map_location=device)
model.eval().to(device)

print(f"n_scales       : {model.n_scales}")
print(f"n_steps_output : {model.n_steps_output}")
print(f"wavelet        : {'enabled' if model.use_wavelet else 'DATA SPACE'}")
print(f"scale shapes   : {model.scale_shapes}")

assert model.n_scales > 1, (
    "Load the 3-scale checkpoint. The probe is vacuous for n_scales=1."
)

In [ ]:
dataset = get_well_dataset(
    path      = well_base_path,
    physics   = well_dataset_name,
    split     = 'test',
    steps     = -1,
    transform = [], #['log_scalars'],
    augment   = [],
)

N_TRAJ = len(dataset.metadata.n_steps_per_trajectory)
print(f"{N_TRAJ} test trajectories, "
      f"{dataset.metadata.n_steps_per_trajectory[0]} steps each")

## 2 · Sensitivity matrix

In [ ]:
N_WINDOWS, N_PROBE, DELTA = 8, 16, 1e-2   # time windows of test trajectory 0; random directions; relative perturbation

item          = dataset[0]
x_full, label = get_well_inputs(item, device=device)

batch = csu.build_probe_batch(
    model,
    x_full        = x_full,
    label         = label,
    n_windows     = N_WINDOWS,     # independent time windows in the batch
    n_steps_input = 4,     # must match training
)
print(batch.summary())

In [ ]:
sens = csu.sensitivity_sweep(
    model,
    batch,
    taus    = (0.1, 0.3, 0.5, 0.7, 0.9),
    delta   = DELTA,        # relative perturbation size
    n_probe = N_PROBE,           # random directions per (tau, source scale)
    seed    = SEED,
)

In [ ]:
# Off-diagonal mass of the tau-averaged sensitivity matrix (paper: 15.1 %).
import json
S = np.asarray(sens["mean"])
off_frac = float((S.sum() - np.trace(S)) / S.sum())
per_tau = [float((M.sum() - np.trace(M)) / M.sum()) for M in np.asarray(sens["matrices"])]
print(np.round(S, 3))
print(f"off-diagonal mass = {100 * off_frac:.1f} %   (determinism floor = {np.mean(sens['floor']):.1e})")
print("per tau:", [f"{100 * r:.1f} %" for r in per_tau])
np.savez(OUT_RAW / "cross_scale_sensitivity_trl_wfm3_haar.npz",
         matrices=sens["matrices"], mean=sens["mean"], taus=sens["taus"], floor=sens["floor"])
RESULT = {"model": "WFM_3scale_haar", "pde": well_dataset_name,
          "mean_matrix": S.round(4).tolist(), "off_diagonal_mass_pct": round(100 * off_frac, 2),
          "per_tau_off_diagonal_pct": [round(100 * r, 2) for r in per_tau],
          "taus": list(map(float, sens["taus"])), "delta": DELTA, "n_probe": N_PROBE,
          "n_windows": N_WINDOWS, "seed": int(SEED), "floor": float(np.mean(sens["floor"]))}
(OUT_RAW / "cross_scale_sensitivity_trl_wfm3_haar.json").write_text(json.dumps(RESULT, indent=1))

In [ ]:
fig = csu.plot_sensitivity(
    sens,
    title    = f"Cross-scale coupling\nTurbulent Radiative Layer 2D",
    savepath = OUT_FIG / "diag_cross_scale_sensitivity_trl.pdf",
)

## 3 · Robustness to the perturbation size

In [ ]:
# Robustness: off-diagonal mass at tau = 0.5 for three perturbation sizes (paper: changes by <= 0.3 points).
rob = {}
for d in (1e-3, 1e-2, 1e-1):
    S_d, floor = csu.sensitivity_matrix(model, batch, tau=0.5, delta=d, n_probe=3)
    rob[f"{d:g}"] = float((S_d.sum() - np.trace(S_d)) / S_d.sum())
    print(f"delta = {d:6.0e}   off-diagonal mass = {rob[f'{d:g}']:6.1%}   floor = {floor:.1e}")
RESULT["delta_sweep_tau0.5_off_diagonal_pct"] = {k: round(100 * v, 2) for k, v in rob.items()}
(OUT_RAW / "cross_scale_sensitivity_trl_wfm3_haar.json").write_text(json.dumps(RESULT, indent=1))